In [181]:
import pandas as pd

df =pd.read_parquet("data.parquet")

## Conversion des types

In [182]:
# domaine_etude => qualitative nominale
# get_dummies drop automatiquement ["domaine_etude"]
# drop_first = Supprime la 1er catégorie (Célibataire) pour éviter la multicolinéarité, elle devient implicite si toutes les autres catégories sont à 0
df = pd.get_dummies(df, columns=["domaine_etude"], drop_first=True)

In [183]:
# statut_marital => qualitative nominale
# get_dummies drop automatiquement ["statut_marital"]
# drop_first = Supprime la 1er catégorie (Célibataire) pour éviter la multicolinéarité, elle devient implicite si toutes les autres catégories sont à 0
df = pd.get_dummies(df, columns=["statut_marital"], drop_first=True)


In [184]:
# departement => qualitative nominale
# get_dummies drop automatiquement ["departement"]
# drop_first = Supprime la 1er catégorie (Commercial) pour éviter la multicolinéarité, elle devient implicite si toutes les autres catégories sont à 0
df = pd.get_dummies(df, columns=["departement"], drop_first=True)


In [185]:
# poste => qualitative nominale
# get_dummies drop automatiquement ["poste"]
# drop_first = Supprime la 1er catégorie (Assistant) pour éviter la multicolinéarité, elle devient implicite si toutes les autres catégories sont à 0
df = pd.get_dummies(df, columns=["poste"], drop_first=True)

In [186]:
pd.DataFrame({
    "colonne": df.columns,
    "type": df.dtypes.values
})

,colonne,type
0,a_quitte_l_entreprise,bool
1,nombre_participation_pee,int64
2,nb_formations_suivies,int64
3,distance_domicile_travail,int64
4,niveau_education,int64
5,frequence_deplacement,int64
6,annees_depuis_la_derniere_promotion,int64
7,annes_sous_responsable_actuel,int64
8,age,int64
9,revenu_mensuel,int64


# séparation train/test

# choix de l’algorithme (régression, classification, clustering…)

Ici nous cherchons à prédire le départ ou non d'un salarié, il s'agit d'une classification binaire

* Régression logistique (LogisticRegression)
* Arbres de décision (DecisionTreeClassifier)
* Forêts aléatoires (RandomForestClassifier)
* Gradient Boosting (XGBoost **, LightGBM, CatBoost, GradientBoostingClassifier)

In [187]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, r2_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from xgboost import XGBClassifier

def entrainement_model(model_name: str) -> None:

    # choix de l’algorithme (régression, classification, clustering…)

    if model_name == "LogisticRegression":
        model = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression())
        ])
    elif model_name == "RandomForestClassifier":
        model = RandomForestClassifier(
            n_estimators=100, # Nombre d’arbres, Plus il y a d’arbres, plus le modèle est puissant, mais aussi plus lent à entraîner
            random_state=42 # Profondeur maximale, trouver le bon équilibre entre la performance et le risque de surapprentissage
        )
    elif model_name == "DecisionTreeClassifier":
        model = DecisionTreeClassifier(
            max_depth=3, 
            random_state=42
        )
    elif model_name == "XGBoost":
        model = XGBClassifier(
            n_estimators=200, 
            max_depth=4, 
            learning_rate=0.1, 
            random_state=42
        )
    else:
        raise ValueError(f"Modèle {model_name} non supporté")

    # Entraînement
    model.fit(X_train, y_train)

    # Prédictions
    y_pred = model.predict(X_test)

    # validation croisée

    # évaluation des performances (accuracy, RMSE, F1, etc.)
    row = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        "r2": r2_score(y_test, y_pred)
    }

    return row


# entraînement du modèle

In [188]:
pass_list = []

# séparation train/test

X = df.drop(columns=['a_quitte_l_entreprise'])
y = df['a_quitte_l_entreprise']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(f"Taille du jeu d'entraînement : {X_train.shape[0]} lignes")
print(f"Taille du jeu de test : {X_test.shape[0]} lignes")

Taille du jeu d'entraînement : 1176 lignes
Taille du jeu de test : 294 lignes


In [189]:
# entrainement de base avec tous les paramètres

results_list = []
results_list.append(entrainement_model("LogisticRegression"))
results_list.append(entrainement_model("RandomForestClassifier"))
results_list.append(entrainement_model("DecisionTreeClassifier"))
results_list.append(entrainement_model("XGBoost"))
pass_list.append(results_list)

In [190]:
# on essaie de supprimer la feature "niveau_hierarchique_poste" (fortement corrélé à "revenu_mensuel")

X_train.drop(columns=["niveau_hierarchique_poste"], inplace=True)
X_test.drop(columns=["niveau_hierarchique_poste"], inplace=True)

results_list = []
results_list.append(entrainement_model("LogisticRegression"))
results_list.append(entrainement_model("RandomForestClassifier"))
results_list.append(entrainement_model("DecisionTreeClassifier"))
results_list.append(entrainement_model("XGBoost"))
pass_list.append(results_list)

In [191]:
# optimisation des hyperparamètres

# Résultat

In [192]:
print("R²")
print("1.0   = (bon modèle)           prédictions exactes ")
print("0.0   = (mauvais modèle)       prédictions proche de la moyenne simple ")
print("< 0.0 = (très mauvais modèle)  prédictions pire que la moyenne ")

for results_list in pass_list:
    results = pd.DataFrame(results_list)
    display(results)

R²
1.0   = (bon modèle)           prédictions exactes 
0.0   = (mauvais modèle)       prédictions proche de la moyenne simple 
< 0.0 = (très mauvais modèle)  prédictions pire que la moyenne 


,model,accuracy,precision,recall,f1,r2
0,LogisticRegression,0.880952,0.576923,0.384615,0.461538,-0.034691
1,RandomForestClassifier,0.874150,0.750000,0.076923,0.139535,-0.093816
2,DecisionTreeClassifier,0.853741,0.166667,0.025641,0.044444,-0.271192
3,XGBoost,0.880952,0.600000,0.307692,0.406780,-0.034691


,model,accuracy,precision,recall,f1,r2
0,LogisticRegression,0.884354,0.592593,0.410256,0.484848,-0.005128
1,RandomForestClassifier,0.877551,0.800000,0.102564,0.181818,-0.064253
2,DecisionTreeClassifier,0.853741,0.166667,0.025641,0.044444,-0.271192
3,XGBoost,0.880952,0.583333,0.358974,0.444444,-0.034691


# Prédictions sur de nouvelles données

# Export

In [193]:
df.to_parquet("model.parquet")